<a href="https://colab.research.google.com/github/pujisubarkah/replikasi_DAiSEE-Towards-User-Engagement-Recognition-in-the-Wild/blob/main/notebook_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import kagglehub

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
FEATURE_ROOT="/content/drive/MyDrive/DAiSEE_Features"

TRAIN_FEATURE=os.path.join(FEATURE_ROOT,"train")
VAL_FEATURE=os.path.join(FEATURE_ROOT,"validation")
TEST_FEATURE=os.path.join(FEATURE_ROOT,"test")

In [ ]:
dataset_path = kagglehub.dataset_download(
    "olgaparfenova/daisee"
)

LABEL_PATH=os.path.join(
    dataset_path,
    "DAiSEE",
    "Labels"
)

train_df=pd.read_csv(
    os.path.join(LABEL_PATH,"TrainLabels.csv")
)

val_df=pd.read_csv(
    os.path.join(LABEL_PATH,"ValidationLabels.csv")
)

test_df=pd.read_csv(
    os.path.join(LABEL_PATH,"TestLabels.csv")
)

In [ ]:
class FeatureDataset(Dataset):

    def __init__(self,dataframe,feature_path,label_column="Engagement"):

        self.df=dataframe.reset_index(drop=True)

        self.feature_path=feature_path

        self.label_column=label_column

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        clip=row["ClipID"]

        clip_id=os.path.splitext(clip)[0]

        feature=np.load(
            os.path.join(
                self.feature_path,
                clip_id+".npy"
            )
        )

        feature=torch.tensor(
            feature,
            dtype=torch.float32
        )

        label=torch.tensor(
            row[self.label_column],
            dtype=torch.long
        )

        return feature,label

In [ ]:
test_dataset=FeatureDataset(
    test_df,
    TEST_FEATURE
)

test_loader=DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm=nn.LSTM(
            input_size=2048,
            hidden_size=256,
            batch_first=True
        )

        self.dropout=nn.Dropout(0.5)

        self.fc=nn.Linear(
            256,
            4
        )

    def forward(self,x):

        out,_=self.lstm(x)

        out=out[:,-1,:]

        out=self.dropout(out)

        out=self.fc(out)

        return out

In [ ]:
device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model=LSTMClassifier().to(device)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/best_model.pth",
        map_location=device
    )
)

model.eval()

In [ ]:
y_true=[]
y_pred=[]

with torch.no_grad():

    for features,labels in test_loader:

        features=features.to(device)

        outputs=model(features)

        preds=torch.argmax(outputs,dim=1)

        y_true.extend(labels.numpy())

        y_pred.extend(preds.cpu().numpy())

In [ ]:
acc=accuracy_score(
    y_true,
    y_pred
)

print(f"Accuracy : {acc:.4f}")

In [ ]:
precision=precision_score(
    y_true,
    y_pred,
    average="weighted"
)

print(f"Precision : {precision:.4f}")

In [ ]:
recall=recall_score(
    y_true,
    y_pred,
    average="weighted"
)

print(f"Recall : {recall:.4f}")

In [ ]:
f1=f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print(f"F1 Score : {f1:.4f}")

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        digits=4
    )
)

In [ ]:
cm=confusion_matrix(
    y_true,
    y_pred
)

print(cm)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(cm)

plt.colorbar()

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.xticks([0,1,2,3])

plt.yticks([0,1,2,3])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            cm[i,j],
            ha="center",
            va="center"
        )

plt.show()